# Understanding Model Inputs and Outputs in Hugging Face

## 1. Input and output concepts

| Concept              | Definition                                                                                      | Example                                                   |
| -------------------- | ----------------------------------------------------------------------------------------------- | --------------------------------------------------------- |
| Input IDs            | Numerical IDs representing the input tokens.                                                    | `[101, 1045, 2293, 9932, 102]`                            |
| Attention Mask       | Identifies real tokens and padding tokens.                                                      | `1` = real token, `0` = padding                           |
| Token Type IDs       | Identify which sentence a token belongs to in models that use segment embeddings, such as BERT. | `0` = first sentence, `1` = second sentence               |
| Tensor Shape         | Describes the dimensions of a tensor.                                                           | `[1, 5]` means one sequence with five tokens              |
| Batch Dimension      | Number of examples processed together.                                                          | `1` means one sentence                                    |
| Sequence Dimension   | Number of tokens in each encoded example.                                                       | `5` means five tokens, including special tokens           |
| Model Output         | The values produced after processing the input.                                                 | Hidden states or classification scores                    |
| Logits               | Raw scores produced by a model before converting them into probabilities.                       | `[2.5, 0.3]`                                              |
| Hidden States        | Numerical representations learned for tokens or intermediate model layers.                      | A 768-dimensional representation for each BERT Base token |
| Output Probabilities | Scores converted into probabilities, usually using softmax for mutually exclusive classes.      | `[0.90, 0.10]`                                            |

## 2. Practical example: Sentiment classification

We will give the sentence `"I love this product"` to a pretrained BERT-based model and predict whether the sentiment is positive or negative.

Python

Run

```
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

text = "I love this product"

# 1. Convert text into model inputs
inputs = tokenizer(text, return_tensors="pt")

print("Input IDs:", inputs["input_ids"])
print("Attention Mask:", inputs["attention_mask"])

if "token_type_ids" in inputs:
    print("Token Type IDs:", inputs["token_type_ids"])
else:
    print("Token Type IDs: Not used by this model")

print("Input Shape:", inputs["input_ids"].shape)
print("Batch Size:", inputs["input_ids"].shape[0])
print("Sequence Length:", inputs["input_ids"].shape[1])

# 2. Run model inference
model.eval()
with torch.no_grad():
    outputs = model(**inputs)

# 3. Inspect logits
logits = outputs.logits
print("Logits:", logits)

# 4. Convert logits to probabilities
probabilities = torch.softmax(logits, dim=-1)
print("Probabilities:", probabilities)

# 5. Get the final prediction
predicted_id = probabilities.argmax(dim=-1).item()
predicted_label = model.config.id2label[predicted_id]

print("Prediction:", predicted_label)

# 6. Inspect hidden states using the base model
from transformers import AutoModel

base_model = AutoModel.from_pretrained(model_name)
base_model.eval()

with torch.no_grad():
    base_outputs = base_model(**inputs)

print("Hidden State Shape:", base_outputs.last_hidden_state.shape)
```

## 3. Logits vs Probability vs Prediction

| Term        | Meaning                                                                | Practical example |
| ----------- | ---------------------------------------------------------------------- | ----------------- |
| Logits      | Raw scores generated by the model. They are not probabilities.         | `[3.2, -1.1]`     |
| Probability | Scores converted into normalized values between 0 and 1 using softmax. | `[0.986, 0.014]`  |
| Prediction  | The final class selected from the model's scores.                      | `POSITIVE`        |

For example, suppose the model produces these illustrative logits:

* Logits: `[3.2, -1.1]`

* After softmax: approximately `[0.987, 0.013]`

* Class labels: `[NEGATIVE, POSITIVE]` would mean the model predicts `NEGATIVE`, because the first score is higher.

The actual label order depends on the model's configuration. Always check `model.config.id2label`.

## 4. Understanding tensor shapes

| Tensor              | Example Shape               | Meaning                                         |
| ------------------- | --------------------------- | ----------------------------------------------- |
| `input_ids`         | `[1, 6]`                    | One example containing six tokens               |
| `attention_mask`    | `[1, 6]`                    | One mask value per token                        |
| `logits`            | `[1, 2]`                    | One example with two class scores               |
| `probabilities`     | `[1, 2]`                    | One probability per class                       |
| `last_hidden_state` | `[1, 6, 768]` for BERT Base | Six token representations, each with 768 values |

The example uses DistilBERT, so its hidden size is 768, but its model architecture and output details differ from BERT Base.

## 5. Important points to remember

* Batch dimension: The first dimension represents the number of examples.

* Sequence dimension: The token count, including special tokens.

* Token type IDs: Only included when the tokenizer/model uses them. DistilBERT generally does not use them.

* Hidden states: Numerical features representing tokens in context; they are not class probabilities.

* Logits: Raw class scores.

* Probability: How the model's scores are distributed across classes.

* Prediction: The selected class based on the model's decision rule.

Interview answer: Input IDs, attention masks, and sometimes token type IDs are passed to the model. The model generates outputs such as hidden states or logits. For classification, logits can be converted into probabilities, and the highest-scoring class is selected as the prediction.


In [ ]:

Input IDs
Attention mask
Token type IDs where applicable
Tensor shape
Batch dimension
Sequence dimension
Model output
Logits
Hidden states
Output probabilities
Explain the difference between:
Logits
Probability
Prediction
Use a practical example.